# Feature engineering

Raw BTS columns -> model features. The model predicts before departure, so anything known only at or after departure is off limits.

Prototype - the final version is in `src/flight_delay/transform/features.py`.

In [8]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(DATA)
flights = df.loc[(df["Cancelled"] == False) & (df["Diverted"] == False), :].copy().reset_index(drop=True)
flights["y"] = flights["ArrDelay"].ge(15).astype(int)
flights.shape

(6965247, 33)

## Column whitelist

I list the allowed columns instead of the forbidden ones, so a new column stays out unless I add it.
`FlightDate` is kept aside for the time split and history features. `Tail_Number` is skipped for now (too many values).

In [9]:
# known before departure
SAFE_RAW = [
    "Month", "DayOfWeek",
    "Reporting_Airline",
    "Origin", "Dest", "OriginState", "DestState",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime",
    "Distance", "DistanceGroup",
]

# known only at / after departure
LEAKAGE = [
    "DepDelay", "DepDelayMinutes", "DepDel15",
    "TaxiOut", "TaxiIn",
    "ActualElapsedTime", "AirTime",
    "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
    "Cancelled", "Diverted", "CancellationCode",
]

assert not (set(SAFE_RAW) & set(LEAKAGE))

# expected: FlightDate, Tail_Number, y
set(flights.columns) - set(SAFE_RAW) - set(LEAKAGE)

{'FlightDate', 'Tail_Number', 'y'}

## Simple features

`CRSDepTime` is HHMM stored as an int (1345 -> 1400 is +55), so I turn it into `DepHour` (2400 -> 0).

`IsWeekend` and `Season` don't add much for XGBoost, but they help the logistic regression baseline.

`Distance` and `DistanceGroup` are basically the same thing.

In [10]:
feat = flights[SAFE_RAW].copy()

# 2400 = midnight
feat["DepHour"] = (flights["CRSDepTime"] // 100).replace(24, 0)

feat["IsWeekend"] = flights["DayOfWeek"].isin([6, 7]).astype(int)

feat["Season"] = flights["Month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "fall", 10: "fall", 11: "fall",
})

feat["y"] = flights["y"]
feat.head()

,Month,DayOfWeek,Reporting_Airline,Origin,Dest,OriginState,DestState,CRSDepTime,CRSArrTime,CRSElapsedTime,Distance,DistanceGroup,DepHour,IsWeekend,Season,y
0,1,1,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
1,1,2,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
2,1,3,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
3,1,4,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,1
4,1,5,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0


## Categorical columns

`Origin`, `Dest` and `Reporting_Airline` have lots of levels. Instead of one-hot I just set them to `category` and use XGBoost's native categorical support (`enable_categorical=True`).

In [11]:
CATEGORICAL = ["Origin", "Dest", "OriginState", "DestState", "Reporting_Airline", "Season"]
feat[CATEGORICAL] = feat[CATEGORICAL].astype("category")

feat.dtypes

Month                    int8
DayOfWeek                int8
Reporting_Airline    category
Origin               category
Dest                 category
OriginState          category
DestState            category
CRSDepTime              int16
CRSArrTime              int16
CRSElapsedTime        float32
Distance              float32
DistanceGroup            int8
DepHour                 int16
IsWeekend               int64
Season               category
y                       int64
dtype: object

## Historical delay rates

For each carrier / airport / route: what share of its flights were delayed before this day. This is basically target encoding, so it can only use earlier days, otherwise it leaks the label.

For a flight on day $$t$$:
$$h_{c,t} = \dfrac{\sum_{s<t}\ \sum_{i \in c,\ \text{date}=s} y_i}{\sum_{s<t}\ \sum_{i \in c} 1}$$

Rare categories get noisy rates, so I pull them toward the global mean with strength $$m$$:
$$\bar y_c = \dfrac{n_c\,\hat p_c + m\,\bar y_{\text{global}}}{n_c + m}$$
A new category ($$n_c = 0$$) just gets the global mean, no NaN.

Computing it on the whole dataset is fine with a time split - test rows only look at earlier days. The global mean is computed on all data, but that leak is negligible.

Implementation: aggregate per `(key, date)`, cumsum over dates, subtract the current day, merge back.

In [ ]:
import numpy as np

PRIOR = float(flights["y"].mean())
M = 100  


def add_history_feature(frame, keys, m=M, prior=PRIOR):
    """Leakage-safe historical delay rate for `keys`, using only days strictly before each flight.

    Smoothed toward `prior` with strength `m`. Returns a float32 array aligned to `frame`'s rows.
    `keys`: a column name or a list (e.g. "Origin", or ["Origin", "Dest"] for a route).
    """
    if isinstance(keys, str):
        keys = [keys]
    daily = (
        frame.groupby(keys + ["FlightDate"], observed=True)["y"]
        .agg(sum_y="sum", n="count")
        .reset_index()
        .sort_values("FlightDate")
    )
    g = daily.groupby(keys, observed=True)
    daily["hist_sum"] = g["sum_y"].cumsum() - daily["sum_y"]
    daily["hist_n"] = g["n"].cumsum() - daily["n"]
    daily["hist_rate"] = (daily["hist_sum"] + m * prior) / (daily["hist_n"] + m)
    out = frame[keys + ["FlightDate"]].merge(
        daily[keys + ["FlightDate", "hist_rate"]], on=keys + ["FlightDate"], how="left"
    )
    return out["hist_rate"].to_numpy(dtype="float32")


feat["CarrierHist"] = add_history_feature(flights, "Reporting_Airline")
feat["OriginHist"] = add_history_feature(flights, "Origin")
feat["RouteHist"] = add_history_feature(flights, ["Origin", "Dest"])

feat[["CarrierHist", "OriginHist", "RouteHist"]].describe()

,CarrierHist,OriginHist,RouteHist
count,6.965247e+06,6.965247e+06,6.965247e+06
mean,2.158489e-01,2.160503e-01,2.149356e-01
std,4.299799e-02,4.143675e-02,5.361684e-02
min,4.898693e-02,5.718584e-02,5.831360e-02
25%,1.888810e-01,1.871302e-01,1.773074e-01
50%,2.115710e-01,2.094301e-01,2.108241e-01
75%,2.419452e-01,2.411077e-01,2.470757e-01
max,3.568850e-01,4.102055e-01,4.922233e-01


## Airport x hour

- `OriginHourCongestion` - number of departures scheduled from the airport in that hour on that day. Schedule only, so no leakage. EDA2 showed that busy hub hours have more delays.
- `OriginHourHist` - historical delay rate for airport + hour, e.g. "ATL at 6pm".

Both need `DepHour`, so it's added to `flights` first.

In [ ]:
flights["DepHour"] = (flights["CRSDepTime"] // 100).replace(24, 0)

feat["OriginHourCongestion"] = (
    flights.groupby(["Origin", "FlightDate", "DepHour"], observed=True)["DepHour"]
    .transform("size")
    .to_numpy(dtype="int32")
)


feat["OriginHourHist"] = add_history_feature(flights, ["Origin", "DepHour"])

feat[["OriginHourCongestion", "OriginHourHist"]].describe()

,OriginHourCongestion,OriginHourHist
count,6.965247e+06,6.965247e+06
mean,2.362590e+01,2.161583e-01
std,2.040283e+01,7.657205e-02
min,1.000000e+00,4.108448e-02
25%,7.000000e+00,1.595149e-01
50%,1.900000e+01,2.093332e-01
75%,3.300000e+01,2.651953e-01
max,1.000000e+02,5.344006e-01


## Leakage check

Checks that no `LEAKAGE` column is in the features and recomputes one carrier's history by hand to make sure only earlier days are used.

In [ ]:
assert not (set(feat.columns) & set(LEAKAGE)), "a leakage column reached the features!"

c = "9E"
sub = flights.loc[flights["Reporting_Airline"] == c, ["FlightDate", "y"]]
dates = np.sort(sub["FlightDate"].unique())

first_mask = (flights["Reporting_Airline"] == c) & (flights["FlightDate"] == dates[0])
assert np.allclose(feat.loc[first_mask, "CarrierHist"].unique(), np.float32(PRIOR))

d = dates[20]
past = sub.loc[sub["FlightDate"] < d, "y"]
expected = (past.sum() + M * PRIOR) / (len(past) + M)
got = feat.loc[(flights["Reporting_Airline"] == c) & (flights["FlightDate"] == d), "CarrierHist"].iloc[0]
assert np.isclose(got, expected, atol=1e-4), (got, expected)

print("leakage guard passed: no post-flight columns, and history uses only the past.")

leakage guard passed: no post-flight columns, and history uses only the past.


## Final matrix

`X`, `y`, and `FlightDate` kept separately (used for the split, not as a feature).

I keep both the raw categories and their history rates - they carry different information. No imputation, XGBoost handles NaN on its own.

In [ ]:
flight_date = flights["FlightDate"]

X = feat.drop(columns="y")
y = feat["y"]

print("X:", X.shape, "| y positive rate:", round(float(y.mean()), 4))
print(X.isna().sum().loc[lambda s: s > 0])
X.dtypes

X: (6965247, 20) | y positive rate: 0.2082

missing per column (XGBoost handles NaN natively - learns a default direction, so we do NOT impute):
Series([], dtype: int64)


Month                       int8
DayOfWeek                   int8
Reporting_Airline       category
Origin                  category
Dest                    category
OriginState             category
DestState               category
CRSDepTime                 int16
CRSArrTime                 int16
CRSElapsedTime           float32
Distance                 float32
DistanceGroup               int8
DepHour                    int16
IsWeekend                  int64
Season                  category
CarrierHist              float32
OriginHist               float32
RouteHist                float32
OriginHourCongestion       int32
OriginHourHist           float32
dtype: object

## Check against the module

`build_features` from `src/` should give the same matrix as this notebook. The only difference: here `Season` is cast to `category`, in `src/` that happens in `prepare_for_xgboost`, so I align that one dtype before comparing.

In [ ]:
from flight_delay.transform import features as F

X_mod, y_mod, date_mod = F.build_features(flights)

X_cmp = X_mod.copy()
X_cmp["Season"] = X_cmp["Season"].astype(X["Season"].dtype)

assert list(X_cmp.columns) == list(X.columns), "column set or order drifted"
pd.testing.assert_frame_equal(X_cmp, X, check_dtype=True)
assert (y_mod == y.to_numpy()).all(), "label drifted"
assert date_mod.equals(flight_date), "flight_date drifted"

print("build_features reproduces this notebook:", X_mod.shape)